# Autoencoder

In [ ]:
import torch
import torch.nn as nn
from utils import StaticGratingsDataset
from modutil import CustomDataset, train

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=2, stride=2),  # 100 -> 50 or 250 -> 125
            nn.LeakyReLU(0.1),
            nn.Conv2d(8, 16, kernel_size=2, stride=2), # 50 -> 25 or 125 -> 62
            nn.LeakyReLU(0.1),
            nn.Conv2d(16, 32, kernel_size=2, stride=2), # 25 -> 12 or 62 -> 31
            nn.LeakyReLU(0.1),
            nn.Flatten(),
            nn.Linear(32 * 12 * 12, latent_dim)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 12 * 12),
            nn.LeakyReLU(0.1),
            nn.Unflatten(1, (32, 12, 12)),
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2, output_padding=1), # 31 -> 62
            nn.LeakyReLU(0.1),
            nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2),  # 62 -> 125
            nn.LeakyReLU(0.1),
            nn.ConvTranspose2d(8, 1, kernel_size=2, stride=2),  # 125 -> 250
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

In [ ]:
sg_dataset = StaticGratingsDataset(750332458)
sg_dataset

In [ ]:
X, y = sg_dataset.get_data(unit_ids=[951826934, 951826887, 951826803, 951811034, 951819869, 951815497, 951811243, 951827248, 951819918, 951810907, 951810817, 951826986, 951815373, 951815505, 951807346, 951823785, 951819732, 951815221, 951826745, 951820184, 951826939, 951820022, 951820288, 951819951, 951824009, 951816276, 951810870, 951820193, 951810625, 951810787, 951810647, 951826923, 951815556, 951819896, 951807510, 951826996, 951819532, 951819971, 951815709, 951827008, 951820079, 951827389, 951810901, 951826969, 951810560, 951826671, 951826756, 951827434, 951820071, 951827261, 951810855, 951810512, 951815788, 951820492, 951816483, 951826853, 951815212, 951815653, 951826875, 951810894, 951826908, 951815488, 951815565, 951826957, 951826870, 951815760, 951815635, 951819927, 951823518, 951823673, 951808263, 951823933, 951810916, 951815841, 951811089, 951816492, 951819744, 951815673, 951820212, 951815700, 951827164, 951820007, 951827752, 951819960, 951827171, 951810931, 951820315, 951815625, 951823553, 951823946, 951823512, 951819769, 951807329, 951810997, 951823605, 951815078, 951820369, 951823624, 951815256, 951807470],stimulus_type="params")
X.shape, y.shape

In [ ]:
y_tensor = torch.from_numpy(y).float()[:,None,:,:]
y_tensor.shape

In [ ]:
dataset = CustomDataset(y_tensor,y_tensor)
model = ConvAutoencoder()
model.to("mps")
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)
def sparse_loss_MSE(recon, target):
    mask = (target > 0).float()
    return (nn.MSELoss(reduction="none")(recon,target) * (1 + 5 * mask)).mean()

In [ ]:
train(model,optimizer,sparse_loss_MSE,None,dataset,None,20,16,device="mps")

In [ ]:
model(y_tensor[0][None,:,:,:].to("mps"))

In [ ]:
y_tensor[0][None,:,:,:]